# 03 – Túlkun með SHAP

Random forest spáir vel fyrir um flokksval en **hvers vegna**? Líkanið lítur ekki aðeins á stakar breytur heldur líka á **samspil þeirra** (t.d. „ungur + háskólamenntaður + hægri/vinstri = ...") og þess vegna er það oft kallað **svartur kassi** (e. *black box*).

[**SHAP**](https://shap.readthedocs.io/) (*SHapley Additive exPlanations*) er aðferð úr leikjafræði sem metur **framlag hverrar breytu** til hverrar einstakrar forspár. Fyrir tréreiknirit er til sérhæft reiknirit (`TreeExplainer`) sem reiknar SHAP-gildi **á nákvæman hátt** og hratt.

Í þessu skjali gerum við eftirfarandi:

1. Hlöðum `rf_base.pkl` og forvinnsluleiðslunni úr öðru skjalinu.
2. Reiknum SHAP-gildi á prófunarhlutanum.
3. Reiknum **heildarmikilvægi breyta** (*global feature importance*).
4. Sýnum **flokkasérhæfða** túlkun.
5. Berum við `rf_enriched` til að sýna hvað gerist þegar málefnaeign er með.

**Niðurlag:** `figures/shap_summary.png`, `figures/shap_summary_per_class.png`, og til samanburðar `figures/shap_summary_enriched.png`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = PROJECT_ROOT / "data" / "processed"
DATA_PATH = PROJECT_ROOT / "Kjósendakönnun" / "ÍSKOS kjósenda 2024" / "ISKOS2024_med_aukabreytum_20260318.sav"
MODELS = PROJECT_ROOT / "models"
FIGURES = PROJECT_ROOT / "figures"

# Load artifacts from notebook 02
rf_base = joblib.load(MODELS / "rf_base.pkl")
rf_enriched = joblib.load(MODELS / "rf_enriched.pkl")

# Train/test data + metadata from notebook 01
test_df = pd.read_parquet(PROC / "test.parquet")
train_df = pd.read_parquet(PROC / "train.parquet")
with (PROC / "feature_groups.json").open() as f:
    fg = json.load(f)
with (PROC / "target_labels.json").open() as f:
    tl = json.load(f)

BASE_FEATURES = fg["base_features"]
ENRICHED_FEATURES = fg["enriched_features"]
y_test = test_df["y"]

print(f"RF base classes      : {list(rf_base.classes_)}")
print(f"RF enriched classes  : {list(rf_enriched.classes_)}")
print(f"Test rows            : {len(test_df):,}")

# EDIT-2026-04-13 imports
import sys
_SRC = str((PROJECT_ROOT / "src").resolve())
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)
from iskos_labels import IskosLabels  # noqa: E402
labels = IskosLabels.from_sav(DATA_PATH, feature_groups_path=PROC / "feature_groups.json")
print(f"Loaded {len(labels.variable_labels)} variable labels.")

## 1 — Reikna SHAP-gildi fyrir `rf_base`

Forvinnsluleiðslan (`prep`) innan leiðslunnar breytir nafnbreytum í one-hot kóðun og skiptir út tölulegum gildum sem vantar fyrir miðgildi. Við þurfum því að gefa `TreeExplainer` **forunnin** gögn, ekki upphafstöfluna.

Til að fá læsilegan breytulista notum við `get_feature_names_out()`.

In [ ]:
# Extract the fitted pieces from the trained Pipeline
prep_base = rf_base.named_steps["prep"]
clf_base = rf_base.named_steps["clf"]

# Transform the TEST set with the already-fit preprocessor
X_test_transformed = prep_base.transform(test_df[BASE_FEATURES])
feature_names_base = prep_base.get_feature_names_out()

# shap.TreeExplainer works directly on the trained sklearn classifier.
# For multi-class RandomForest, shap_values returns a list of arrays (one per class).
explainer_base = shap.TreeExplainer(clf_base)
shap_values_base = explainer_base.shap_values(X_test_transformed)

# In recent shap versions, shap_values can return a 3D numpy array
# with shape (n_samples, n_features, n_classes). Normalize that here.
if isinstance(shap_values_base, np.ndarray) and shap_values_base.ndim == 3:
    shap_values_base_list = [shap_values_base[:, :, i] for i in range(shap_values_base.shape[2])]
else:
    shap_values_base_list = shap_values_base
print(f"Number of classes in SHAP output: {len(shap_values_base_list)}")
print(f"Shape per class: {shap_values_base_list[0].shape}")
print(f"(n_test_samples={X_test_transformed.shape[0]}, n_features={X_test_transformed.shape[1]})")

## 2 — Heildarmikilvægi breyta

Þessi mynd sýnir **meðaltal algilds SHAP-gildis** fyrir hverja breytu, summað yfir alla flokkana sjö. Hærri stika þýðir að breytan hafi meiri áhrif á forspána almennt (óháð því í hvaða átt).

In [ ]:
# EDIT-2026-04-13 summary-readable
# Map every (possibly one-hot) feature to its Icelandic short label
feature_names_readable_base = [
    labels.readable(name) for name in feature_names_base
]

# Global bar plot (mean |SHAP| per feature, aggregated across classes)
fig = plt.figure(figsize=(10, 9))
shap.summary_plot(
    shap_values_base_list,
    X_test_transformed,
    feature_names=feature_names_readable_base,
    class_names=list(clf_base.classes_),
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.title("SHAP heildarmikilvægi – random forest á grunnbreytum")
plt.tight_layout()
fig.savefig(FIGURES / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 — Flokkasérhæfð túlkun

Hver breyta hefur mismunandi áhrif eftir því **í hvaða átt** hún ýtir forspánni. Til að skilja það skoðum við *býflugnamynd* (e. *beeswarm plot*) fyrir hvern flokk: hver punktur er einn aðili sem svarar könnuninni en hann er staðsettur á lárétta ásnum eftir SHAP-gildinu (hvort breytan ýtir forspánni **í átt að** eða **frá** viðkomandi flokki) og litur sýnir **gildi breytunnar** (rauður = hátt gildi, blár = lágt).

In [ ]:
# EDIT-2026-04-13 per-class-tall
def plot_beeswarm_per_class(shap_list, X, feat_names, classes, out_path: Path, max_display=12):
    '''One beeswarm subplot per class, with very generous vertical room.'''
    n_classes = len(shap_list)
    # Each subplot needs ~0.55 inches per displayed feature, plus title room.
    per_subplot_height = max(16.5, max_display * 10.6 + 1.5)
    fig_height = per_subplot_height * n_classes
    fig, axes = plt.subplots(n_classes, 1, figsize=(12, fig_height))
    if n_classes == 1:
        axes = [axes]
    for i, (sv, cls_code) in enumerate(zip(shap_list, classes)):
        plt.sca(axes[i])
        shap.summary_plot(
            sv, X, feature_names=feat_names, show=False,
            max_display=max_display, plot_type="dot",
        )
        title_party = tl["target_class_labels_is"].get(cls_code, cls_code)
        axes[i].set_title(f"SHAP – flokkur: {title_party}", loc="left", fontsize=12)
    # Generous vertical spacing — beeswarms are dense, they need room
    plt.subplots_adjust(hspace=0.9)
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

# Use readable feature names so each y-tick is human-meaningful
feature_names_readable_base = [labels.readable(n) for n in feature_names_base]
plot_beeswarm_per_class(
    shap_values_base_list,
    X_test_transformed,
    feature_names_readable_base,
    list(clf_base.classes_),
    FIGURES / "shap_summary_per_class.png",
    max_display=12,
)

## 4 — Samanburður: hvað gerist þegar málefnaeign er með?

Nú endurtökum við sömu greiningu fyrir **útvíkkaða líkanið** (sem fékk einnig `bprt_*`-breyturnar um hvaða flokki svarendur treysta best). Tilgangur: sjá hvernig þessar útkomu breytur **yfirgnæfa** hinar.

Þetta er bein birtingarmynd kaflaumfjöllunarinnar um **sýndarsamhengi** (*spurious correlation*) og **orsakavs.**-tengsl í vélnámi.

In [ ]:
# EDIT-2026-04-13 enriched-readable
prep_enr = rf_enriched.named_steps["prep"]
clf_enr = rf_enriched.named_steps["clf"]
X_test_enr = prep_enr.transform(test_df[ENRICHED_FEATURES])
feat_names_enr = prep_enr.get_feature_names_out()

explainer_enr = shap.TreeExplainer(clf_enr)
shap_values_enr = explainer_enr.shap_values(X_test_enr)
if isinstance(shap_values_enr, np.ndarray) and shap_values_enr.ndim == 3:
    shap_values_enr_list = [shap_values_enr[:, :, i] for i in range(shap_values_enr.shape[2])]
else:
    shap_values_enr_list = shap_values_enr

fig = plt.figure(figsize=(9, 8))
shap.summary_plot(
    shap_values_enr_list,
    X_test_enr,
    feature_names=[labels.readable(n) for n in feat_names_enr],
    class_names=list(clf_enr.classes_),
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.title("SHAP heildarmikilvægi – random forest á útvíkkuðu breytum")
plt.tight_layout()
fig.savefig(FIGURES / "shap_summary_enriched.png", dpi=150, bbox_inches="tight")
plt.show()

## 5 — Af hverju ekki SHAP á AutoGluon?

AutoGluon býr til **staflað safn** (*stacked ensemble*) úr mörgum ólíkum líkönum: tréreiknirit, gradient boosting, nálægðarlíkön, taugarnet. SHAP hefur enga nákvæma útfærslu fyrir slíkar samsetningar. Beita þyrfti líkindafræðilegri nálgun (`KernelExplainer`) sem er **hæg og ónákvæm** þegar
breyturnar eru margar.

Þetta er **ein algengasta takmörkun sjálfvirks vélnáms**: þegar markmiðið er túlkun en ekki bara nákvæmni er oft betra að nota einfaldari og gegnsærri líkön.

## 6 — Tafla: topp 15 breytur fyrir kaflann

Að lokum útbúum við töflu yfir topp 15 breytur samkvæmt heildar SHAP-gildi.

In [ ]:
# EDIT-2026-04-13 top-features-readable
def top_features(shap_list, feat_names, k=15):
    stacked = np.stack([np.abs(sv) for sv in shap_list])   # (n_classes, n_samples, n_feats)
    mean_abs = stacked.mean(axis=(0, 1))                    # mean over classes AND samples
    order = np.argsort(mean_abs)[::-1][:k]
    return pd.DataFrame({
        "feature_code": [feat_names[i] for i in order],
        "feature": [labels.readable(feat_names[i]) for i in order],
        "mean_abs_shap": mean_abs[order].round(4),
    })

top_base = top_features(shap_values_base_list, feature_names_base, k=15)
top_enr = top_features(shap_values_enr_list, feat_names_enr, k=15)

top_base.to_csv(FIGURES / "top_features_rf_base.csv", index=False)
top_enr.to_csv(FIGURES / "top_features_rf_enriched.csv", index=False)

print("Topp 15 breytur á grunnlíkani:")
print(top_base.to_string(index=False))
print("\nTopp 15 breytur á útvíkkuðu líkani:")
print(top_enr.to_string(index=False))

## 7 — SHAP-túlkun á leiðréttu líkani (*balanced*)

Í öðru skjalinu þjálfuðum við **annað random forest** þar sem við vógum litlu flokkana upp (`class_weight="balanced"`). Stærstu breytingar sáust í **flokkasérhæfðri** endurheimt en líkanið finnur litlu flokkana miklu betur.

Spurning: **notar leiðrétta líkanið sömu breytur og fyrri útgáfan?** Eða reiðir það sig á aðrar breytur til að greina litlu flokkana? Við reiknum SHAP-gildi á `rf_base_balanced` og berum beint saman við naive-líkanið.

In [ ]:
# Load the balanced counterpart
rf_base_balanced = joblib.load(MODELS / "rf_base_balanced.pkl")
prep_bal = rf_base_balanced.named_steps["prep"]
clf_bal = rf_base_balanced.named_steps["clf"]

X_test_bal = prep_bal.transform(test_df[BASE_FEATURES])
feat_names_bal = prep_bal.get_feature_names_out()

# Explain the balanced model
explainer_bal = shap.TreeExplainer(clf_bal)
shap_values_bal = explainer_bal.shap_values(X_test_bal)

# Normalize shape to list-of-arrays (one per class) for consistency with section 1
if isinstance(shap_values_bal, np.ndarray) and shap_values_bal.ndim == 3:
    shap_values_bal_list = [shap_values_bal[:, :, i] for i in range(shap_values_bal.shape[2])]
else:
    shap_values_bal_list = shap_values_bal

print(f"Balanced model classes: {list(clf_bal.classes_)}")
print(f"SHAP array shape per class: {shap_values_bal_list[0].shape}")

In [ ]:
# EDIT-2026-04-13 balanced-readable
# Global bar plot for the balanced model
fig = plt.figure(figsize=(9, 8))
shap.summary_plot(
    shap_values_bal_list,
    X_test_bal,
    feature_names=[labels.readable(n) for n in feat_names_bal],
    class_names=list(clf_bal.classes_),
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.title("SHAP heildarmikilvægi – random forest (grunn, balanced)")
plt.tight_layout()
fig.savefig(FIGURES / "shap_summary_balanced.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 — Samanburður: *naive* vs. *balanced* (topp breytur)

Hér koma tvær myndir hlið við hlið, ein fyrir naive-líkanið og önnur fyrir leiðrétta líkanið, og tafla sem sýnir **breytingar á röð breyta** milli
umferðanna. Þegar leiðrétta líkanið notar aðrar breytur er það **vísbending um að fyrri umferðin hafi verið að reyna að ná í stóra flokkana**, en leiðrétta umferðin reynir í staðinn að greina litlu flokkana með öðrum vísbendingum.

Þetta er lykilþáttur kaflans: **hvað ML „lærir" er háð því hvernig við stilllum þjálfunina**, ekki bara gögnin sjálf.

In [ ]:
# EDIT-2026-04-13 naive-vs-balanced-readable
# Helper: compute mean |SHAP| per feature from a list-of-arrays shap output
def mean_abs_shap(shap_list, feat_names):
    stacked = np.stack([np.abs(sv) for sv in shap_list])
    return pd.Series(stacked.mean(axis=(0, 1)), index=feat_names)

mabs_naive = mean_abs_shap(shap_values_base_list, feature_names_base).sort_values(ascending=False)
mabs_balanced = mean_abs_shap(shap_values_bal_list, feat_names_bal).sort_values(ascending=False)

# Combined table: union of top-k from both passes, with ranks from each
K = 15
union = list(dict.fromkeys(list(mabs_naive.head(K).index) + list(mabs_balanced.head(K).index)))

naive_rank = {f: i + 1 for i, f in enumerate(mabs_naive.index)}
bal_rank = {f: i + 1 for i, f in enumerate(mabs_balanced.index)}

comparison = pd.DataFrame({
    "feature_code": union,
    "feature": [labels.readable(f) for f in union],
    "mean_abs_shap_naive": [mabs_naive.get(f, 0.0) for f in union],
    "mean_abs_shap_balanced": [mabs_balanced.get(f, 0.0) for f in union],
    "rank_naive": [naive_rank.get(f, None) for f in union],
    "rank_balanced": [bal_rank.get(f, None) for f in union],
}).round(4)
comparison["rank_change"] = comparison["rank_naive"] - comparison["rank_balanced"]  # +ve = balanced ranks it higher

comparison.to_csv(FIGURES / "top_features_naive_vs_balanced.csv", index=False)
print("Topp breytur — naive vs balanced (raðbreyting: jákvætt = balanced metur hana mikilvægari):")
print(comparison.to_string(index=False))

In [ ]:
# Side-by-side bar chart of top 15 features in each pass
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=False)

top_naive = mabs_naive.head(15).iloc[::-1]  # reverse so biggest is at top
top_bal = mabs_balanced.head(15).iloc[::-1]

axes[0].barh(top_naive.index, top_naive.values, color="#c75b5b")
axes[0].set_title("Naive: mean |SHAP|")
axes[0].set_xlabel("Mean |SHAP|")
axes[0].tick_params(axis="y", labelsize=9)

axes[1].barh(top_bal.index, top_bal.values, color="#4a90a4")
axes[1].set_title("Balanced: mean |SHAP|")
axes[1].set_xlabel("Mean |SHAP|")
axes[1].tick_params(axis="y", labelsize=9)

fig.suptitle("Topp 15 breytur eftir SHAP — naive vs balanced")
plt.tight_layout()
fig.savefig(FIGURES / "shap_naive_vs_balanced.png", dpi=150, bbox_inches="tight")
plt.show()

## Samantekt

| Skrá | Innihald |
|---|---|
| `figures/shap_summary.png` | Heildarmikilvægi breyta (naive grunnlíkan) – Mynd 2 í kaflanum |
| `figures/shap_summary_per_class.png` | Flokkasérhæfð túlkun fyrir alla sjö flokkana |
| `figures/shap_summary_enriched.png` | Útvíkkað líkan með málefnaeign |
| `figures/shap_summary_balanced.png` | Leiðrétt líkan (`rf_base_balanced`) |
| `figures/shap_naive_vs_balanced.png` | Samanburðarmynd hlið við hlið |
| `figures/top_features_rf_base.csv` | Topp 15 breytur (naive) |
| `figures/top_features_rf_enriched.csv` | Topp 15 breytur í útvíkkuðu líkani |
| `figures/top_features_naive_vs_balanced.csv` | Raðbreytingar milli naive og balanced |

**Verkefni kláruð.** Kaflinn hefur nú allt sem þarf til að fylla inn XX,X%
gildin, útskýra hlutdrægni fyrri umferðar, og rökstyðja leiðrétta umferðina.